# Convert EarthCARE lightning datasets from NetCDF to Parquet
This notebook reads NetCDF files from the EarthCARE lightning datasets, and converts them into Parquet format for efficient storage and access. 
- Monthly files are created for the EarthCARE-frame lightning collection, separately for the GLM and LI instruments. 
- For along-track lightning counts, one parquet file is created for GLM and one for LI. 
- For the lightning storm catalogue, two parquet files are produced: one containing cluster summaries and one containing the time evolution of lightning clusters.

In [ ]:
import glob
import xarray as xr
import geopandas as gpd
import os
import re
from collections import defaultdict
import pandas as pd
from pathlib import Path
import gc

In [ ]:
source_dir = '/path/to/data/'
output_dir = '/output/dir/'

files = sorted(glob.glob(source_dir + 'lightning_groups/*.nc'))
files2 = sorted(glob.glob(source_dir + 'track_counts/*.nc'))

In [ ]:
def yield_files_by_month(file_list, date_pattern):

    # Dictionary to hold the groups: Key = (2024, 8), Value = [file1, file2...]
    grouped_files = defaultdict(list)

    for file_path in file_list:
        filename = os.path.basename(file_path)
        match = date_pattern.search(filename)
        
        if match:
            year = int(match.group(1))
            month = int(match.group(2))
            grouped_files[(year, month)].append(file_path)
        else:
            print(f"Warning: Could not parse date from {filename}")

    for (year, month), files in grouped_files.items():
        yield (year, month), files

In [ ]:
def convert_nc_to_parquet(file_list, columns):
    """
    Reads a list of EarthCARE NC files, converts them to a GeoDataFrame,
    sorts by time and location, and saves as GeoParquet.
    """
    dataframes = []

    print(f"Processing {len(file_list)} files...")

    for file_path in file_list:
        try:
            # Open the NetCDF file
            with xr.open_dataset(file_path, decode_timedelta=True) as ds:

                # Convert to Pandas DataFrame
                subset = ds[columns]
                df = subset.to_dataframe().reset_index(drop=True)

                # Extract EarthCARE ID
                path_obj = Path(file_path)
                earthcare_id = path_obj.stem.split('_')[-1]                
                df['earthcare_id'] = earthcare_id

                # add to list
                dataframes.append(df)

        except Exception as e:
            print(f"Error processing file {file_path}: {e}")


    print("Concatenating data...")
    full_df = pd.concat(dataframes, ignore_index=True)

    del dataframes
    gc.collect()


    print("Converting to GeoDataFrame...")
    # Create the geometry column from coordinates
    geometry = gpd.points_from_xy(full_df['longitude'], full_df['latitude'])
    gdf = gpd.GeoDataFrame(full_df, geometry=geometry, crs="EPSG:4326")

    del full_df
    gc.collect()

    print("Sorting data...")
    gdf.sort_values(
        by=['geometry', 'group_time'], 
        inplace=True, 
        ignore_index=True
    )

    return gdf

### Collection 1

Convert and store the GLM files

In [ ]:
parquet_dir = output_dir + 'lightning_groups_parquet/'
source = 'GLM'
date_pattern = re.compile(r"GLM_(\d{4})(\d{2})")
source_files = [f for f in files if source in f]
columns = ['group_id',
 'group_time',
 'latitude',
 'longitude',
 'radiance',
 'flash_id',
 'ec_time_diff',
 'parent_cluster_id',
 'cluster_id',
 'distance_from_nadir']

earthcare_ids = {}

for ((year, month), filelist) in yield_files_by_month(source_files, date_pattern):
    gdf = convert_nc_to_parquet(filelist, columns)
    gdf['source'] = source
    gdf.to_parquet(
        parquet_dir + f'EC_lightning_{source}_{year}_{month}.parquet',
        engine='pyarrow',
        compression='zstd',
        write_covering_bbox=True, 
        schema_version='1.1.0'
    )
    for id in gdf['earthcare_id'].unique():
        earthcare_ids.setdefault(id, []).append(f'EC_lightning_{source}_{year}_{month}.parquet')


Convert and store the LI files

In [ ]:
parquet_dir = output_dir + 'lightning_groups_parquet/'
source = 'LI'
date_pattern = re.compile(r"LI_(\d{4})(\d{2})")
source_files = [f for f in files if source in f]
columns = ['group_id',
 'group_time',
 'latitude',
 'longitude',
 'radiance',
 'flash_id',
 'parallax_corrected_lat',
 'parallax_corrected_lon',
 'ec_time_diff',
 'parent_cluster_id',
 'cluster_id',
 'distance_from_nadir']


for ((year, month), filelist) in yield_files_by_month(source_files, date_pattern):
    gdf = convert_nc_to_parquet(filelist,columns)
    gdf['source'] = source
    gdf.to_parquet(
        parquet_dir + f'EC_lightning_{source}_{year}_{month}.parquet',
        engine='pyarrow',
        compression='zstd',
        write_covering_bbox=True, 
        schema_version='1.1.0'
    )
    
    for id in gdf['earthcare_id'].unique():
        earthcare_ids.setdefault(id, []).append(f'EC_lightning_{source}_{year}_{month}.parquet')

Write the earthcare-id -> file dictionary as a parquet file.

In [ ]:
df = pd.Series(earthcare_ids).reset_index()
df.columns = ['earthcare_id', 'files']
df.to_parquet(output_dir + 'earthcare_id_mapping.parquet', engine='pyarrow', compression='zstd')

### Collection 2

In [ ]:
def process_col2_nc_to_geoparquet(file_list):
    """
    Reads EarthCARE NC files, expands 2D variables (cpr, subcluster_id) 
    into a flat dataframe, sorts, and saves as GeoParquet.
    """
    dataframes = []
    
    for file_path in file_list:
        try:
            with xr.open_dataset(file_path) as ds:

                df = ds.to_dataframe().reset_index()
                path_obj = Path(file_path)
                earthcare_id = path_obj.stem.split('_')[-1]
                df['earthcare_id'] = earthcare_id
                dataframes.append(df)

        except Exception as e:
            print(f"Skipping {file_path}: {e}")


    print("Concatenating all files...")
    full_df = pd.concat(dataframes, ignore_index=True)

    del dataframes
    gc.collect()

    print("Creating GeoDataFrame...")
    geometry = gpd.points_from_xy(full_df['longitude'], full_df['latitude'])
    full_df.drop(columns=['latitude', 'longitude', 'cpr'], inplace=True)
    gdf = gpd.GeoDataFrame(full_df, geometry=geometry, crs="EPSG:4326")

    del full_df
    gc.collect()

    print("Sorting data...")
    gdf.sort_values(
        by=['geometry', 'time'], 
        inplace=True, 
        ignore_index=True
    )

    return gdf

Process LI files in one go.

In [ ]:
parquet_dir = output_dir + 'track_counts_parquet/'
source = 'LI'
source_files = [f for f in files2 if source in f]

gdf = process_col2_nc_to_geoparquet(source_files)
gdf['source'] = source

gdf.to_parquet(
    parquet_dir + f'EC_track_lightning_{source}.parquet',
    engine='pyarrow',
    compression='zstd',
    write_covering_bbox=True, 
    schema_version='1.1.0'
)


Process GLM files in one go.

In [ ]:
parquet_dir = output_dir + 'track_counts_parquet/'
source = 'GLM'
source_files = [f for f in files2 if source in f]

gdf = process_col2_nc_to_geoparquet(source_files)
gdf['source'] = source

gdf.to_parquet(
    parquet_dir + f'EC_track_lightning_{source}.parquet',
    engine='pyarrow',
    compression='zstd',
    write_covering_bbox=True, 
    schema_version='1.1.0'
)

### Collection 3

Process the collection 3 files.

In [ ]:
import json
import pandas as pd
import geopandas as gpd
import numpy as np
from datetime import timedelta

def transform_json_to_parquet(json_file_path):

    with open(json_file_path, 'r') as f:
        content = json.load(f)
    
    data = content['data']

    df_summary = pd.DataFrame(data).drop(
        columns=['minute_counts', 'minute_mean_lat', 'minute_mean_lon', 'masked_minute_counts']
    )
    
    # Convert types
    df_summary['peak_datetime'] = pd.to_datetime(df_summary['peak_datetime'])
    gdf_summary = gpd.GeoDataFrame(
        df_summary,
        geometry=gpd.points_from_xy(df_summary['peak_lon'], df_summary['peak_lat']),
        crs="EPSG:4326"
    )
    
    # the Evolution GeoDataFrame (Detail) ---
    evolution_rows = []
    
    for entry in data:
        common_ids = {
            'unique_id': entry['unique_id'],
            'peak_datetime': pd.to_datetime(entry['peak_datetime'])
        }
        
        minutes = entry['minute_counts'].keys()
        
        for m_str in minutes:
            minute_offset = int(m_str)
            abs_time = common_ids['peak_datetime'] + timedelta(minutes=minute_offset)
        
            row = common_ids.copy()
            row['minute_offset'] = minute_offset
            row['absolute_time'] = abs_time
            row['lightning_count'] = entry['minute_counts'][m_str]
            row['masked_lightning_count'] = entry['masked_minute_counts'][m_str]
            row['latitude'] = entry['minute_mean_lat'][m_str]
            row['longitude'] = entry['minute_mean_lon'][m_str]
            
            evolution_rows.append(row)
            
    df_evolution = pd.DataFrame(evolution_rows)
    
    # Create Geometry (Track Location)
    gdf_evolution = gpd.GeoDataFrame(
        df_evolution,
        geometry=gpd.points_from_xy(df_evolution['longitude'], df_evolution['latitude']),
        crs="EPSG:4326"
    )
    
    gdf_evolution = gdf_evolution.sort_values(by=['unique_id', 'absolute_time',])

    return gdf_summary, gdf_evolution

In [ ]:
json_file_path = source_dir + 'EarthCARE_lightning_storm_catalogue.json'

gdf_summary, gdf_evolution = transform_json_to_parquet(json_file_path)

In [ ]:
gdf_summary.to_parquet(output_dir + 'EC_lightning_clusters.parquet', index=False)
gdf_evolution.to_parquet(output_dir + 'EC_lightning_cluster_evolution.parquet', index=False)

## Tests and Asserts

The following cells test that the parquet output is consistent with the
original NetCDF input by comparing values from a randomly selected file.

In [ ]:
import glob
import xarray as xr
import geopandas as gpd
import numpy as np
import os

files = sorted(glob.glob('/home/bpiskala/Object_Data/lightning_processing/lightning_groups_20240801_20260131/*.nc'))

In [ ]:
def extract_file_info(file_path):
    filename = os.path.basename(file_path)
    parts = filename.split('_')
    source = parts[0]
    date_str = parts[1]
    year = int(date_str[:4])
    month = int(date_str[4:6])
    return source, year, month

np.random.seed(42)
rf = np.random.choice(files)  # Randomly select a file for testing
t, y, m = extract_file_info(rf)
t,y,m

In [ ]:
new = gpd.read_parquet(f'/home/bpiskala/Object_Data/lightning_processing/lightning_groups_parquet/EC_lightning_{t}_{y}_{m}.parquet')
new

In [ ]:
orig = xr.open_dataset(rf, decode_timedelta=True)
orig

In [ ]:
for d in orig.data_vars:

    assert orig[d].dropna(dim='groups').isin(new[d].values).all().values

In [ ]:
new[new.group_id == orig['group_id'].values[0]]

In [ ]:
subset = new[new.group_id.isin(orig['group_id'].values) & new.group_time.isin(orig['group_time'].values)]
subset = subset.sort_values(by='group_id').reset_index(drop=True)
orig = orig.sortby("group_id")


columns_to_check = ['group_id','latitude', 'longitude', 'radiance', 'flash_id', 'parent_cluster_id', 'cluster_id', 'distance_from_nadir']
for c in columns_to_check:
    assert np.isclose(subset[c], orig[c].values, equal_nan=True).all()